In [1]:
from merge_tables.db.connection import connect_to_postgres_via_duckdb

duck = connect_to_postgres_via_duckdb()

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [19]:
duck.sql(
    """
    select 
        a.*,
        coalesce(c."Kontakt: Firma" , bc.firm_name),
        em.easybill_id
    from read_csv('/private/var/folders/r3/svzsdlzn73xd8g34qr5rhw9h0000gq/T/17760908682355/postal_address_0_20260413_140216.elt_all_resources_firm_other_1.transform_and_load_postal_address.clean_incoherent.csv') a
    left join pg.bas_firms.easybill_medisoft em
        on split(firmSourceId, '_')[-1] = em.id
    left join pg.easybill.contacts c
        on c."Kontakt: Kundennummer" = em.easybill_id
    left join pg.bas_firms.full_basic_care bc
        on split(firmSourceId, '_')[-1] = bc.id
    where error_type = 'Suppression'
    """
).to_csv('too_long_addresses.csv')

In [10]:
duck.sql("select * from pg.bas_firms.full_basic_care")

┌───────┬──────────────────┬─────────────────────────────────────────────────────────────────┬───────────┬───────────┐
│  id   │ mother_client_id │                            firm_name                            │ Standort  │ Anschrift │
│ int32 │     varchar      │                             varchar                             │  varchar  │  varchar  │
├───────┼──────────────────┼─────────────────────────────────────────────────────────────────┼───────────┼───────────┤
│     1 │ 100000003        │ Ates & Partner ÜBAG GbR Sivan Ates Banu Mareille Ates Zahnärzte │ Köln      │ NULL      │
│     2 │ 100020000        │ ABZ Ambulantes Betreuungs Zentrum GmbH                          │ Berlin    │ NULL      │
│     3 │ 100020001        │ AUFGEPASST e.V.                                                 │ Berlin    │ NULL      │
│     4 │ 100020006        │ Ambulanter Pflegedienst la vie GmbH                             │ Berlin    │ NULL      │
│     5 │ 100020007        │ allod Immobilien- u